In [0]:
%load_ext autoreload
%autoreload 2

In [0]:

from pyspark.sql import SparkSession
import src.utils.helpers
import src.bronze.cleaning


In [0]:
from dataclasses import dataclass, field

@dataclass(frozen=True)
class BronzePull:
    directory: str
    grain: str
    table_suffix: str
    natural_key: tuple[str, ...]
    file_path: str
    file_format: str = "csv"
    options: dict[str, str] = field(default_factory=lambda: {
        "header": "true",
        "inferSchema": "false",
    })

    @property
    def table_name(self):
        return (
            f"bronze_dev.redfin."
            f"{self.directory}_{self.table_suffix}"
        )

In [0]:
DIRECTORIES = (
    "housing_market",
    "price_drops",
    "delistings_relistings",
)

GRAINS = {
    "counties": "county",
    "zips": "zipcode",
    "neighborhoods": "neighborhood",
}

DEFAULT_KEY = (
    "PERIOD_BEGIN",
    "PERIOD_END",
    "REGION_NAME",
)

data_pulls = [
    BronzePull(
        directory=d,
        grain=g,
        table_suffix=s,
        natural_key=DEFAULT_KEY,
        file_path=f"s3://redfin-public-data/redfin_data_center/"
                  f"{d}/monthly/all_{g}.csv"
    )
    for d in DIRECTORIES
    for g, s in GRAINS.items()
]

In [0]:
for p in data_pulls:
    print(p.file_path)

In [0]:

def create_bronze_table(pull):

    @dp.table(name=pull.table_name)
    def table():
        return src.utils.helpers.read_bronze_dataset(
            pull.file_path,
            pull.file_format,
            pull.table_name,
            spark,
            pull.options
        )

    return table


for pull in data_pulls:
    #print(type(globals()))
    #globals()[pull.table_name.split(".")[-1]] = create_bronze_table(pull)
    globals()[pull.table_name.split(".")[-1]] = 'a'

In [0]:
from pyspark import pipelines as dp